# Context

**Dataset:** Hillstrom / MineThatData email challenge (64,000 customers), via `sklift.datasets.fetch_hillstrom`.

**Design:** 1/3 Mens E-Mail, 1/3 Womens E-Mail, 1/3 No E-Mail (control), randomized.



In [1]:
import pandas as pd
import numpy as np
from sklift.datasets import fetch_hillstrom

pd.set_option('display.max_columns', None)

/Users/chunliu/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## EDA

In [2]:
bunch = fetch_hillstrom(target_col='visit')

df = bunch.data.copy()
df['segment'] = bunch.treatment
df['visit'] = bunch.target

print(df.shape)
df.head()

(64000, 10)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0


In [3]:
df.dtypes

recency              int64
history_segment     object
history            float64
mens                 int64
womens               int64
zip_code            object
newbie               int64
channel             object
segment             object
visit                int64
dtype: object

In [4]:
df.isna().sum()

recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
dtype: int64

### Arm sizes

In [5]:
df['segment'].value_counts()

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

### Collapse to two-arm treatment indicator

the main analysis collapses Mens E-Mail + Womens E-Mail into a single `treatment = 1` ("any email") vs. `treatment = 0` ("No E-Mail").

In [6]:
df['treatment'] = (df['segment'] != 'No E-Mail').astype(int)
df['treatment'].value_counts()

treatment
1    42694
0    21306
Name: count, dtype: int64

### Outcome base rate

In [7]:
df.groupby('treatment')['visit'].agg(['mean', 'count'])

,mean,count
treatment,,
0,0.106167,21306
1,0.167049,42694


### Randomization balance check

This is the core sanity check for this notebook: if randomization worked, the two arms (`treatment` 0 vs. 1) should look statistically indistinguishable on every *pre-treatment* covariate — recency, history, mens/womens purchase history, channel, zip/urban-rural, and newbie status. Any imbalance here is a real RCT-integrity flag (not necessarily fatal, but it must be surfaced), not something to explain away silently.

**Handed off to you from here on** — EDA, the balance tests themselves (what test fits a continuous covariate vs. a categorical one, and why), interpretation, and everything downstream (baseline model, uplift model, evaluation, profit curve, write-up).